In [0]:
import pyspark.sql as ps
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.functions import col, trim
from pyspark.sql.window import Window

#Read from bronze table

In [0]:
df = spark.table("workspace.bronze.crm_prd_info")

#Silver Transformation


##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Product key parsing

In [0]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))


##Cost Cleanup

In [0]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

##Product  Line Normalization

In [0]:
df = (
    df
    .withColumn(
        "prd_line"
        , F.when(col("prd_line") == "M" , "Mountain")
            .when(col("prd_line") == "S", "Other Sales")
            .when(col("prd_line") == "T", "Touring")
            .when(col("prd_line") == "R", "Road")
            . otherwise("n/a")

    )
)

##Date Casting

In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))


##Renaming Column

In [0]:
Rename_Map = {
    "prd_id": "product_id",
    "prd_key": "product_Number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date",
    "cat_id": "category_id"
}

for old_name , new_name in Rename_Map.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity Check of dataframe

In [0]:
df.limit(10).display()

#Writing dataframe to silver table

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.crm_products")

##Sanity Check of silver table

In [0]:
%sql
select count(*) from workspace.silver.crm_products